# BRFSS simple baseline notebook

**Downstream task**: Binary classification — dự đoán **có bệnh tim / không có bệnh tim** trên BRFSS, tập **intersection 2019–2021**.

## End-to-end workflow (theo 4 bước)
1. **Task**: heart disease (`_MICHD` target chính, fallback `CVDINFR4 | CVDCRHD4`).
2. **Input features**: enforce **intersection across years** — chỉ giữ canonical feature resolve được ở mọi năm load. Hiện candidate set có **18 features** (15 cũ + 3 dietary từ paper jcdd-11-00396: `Fruit_Consumption`, `Green_Vegetables_Consumption`, `FriedPotato_Consumption`).
3. **Model + paradigm**: 4 baseline supervised: `logreg`, `xgb`, `catboost`, `tabtransformer`.
4. **Train + evaluate**: tự động; output JSON + CSV vào `./baseline_notebook_outputs/`.

## Key design decisions
- **Preprocessing**: decode raw codes → canonical, lọc `DISPCODE ∈ {110, 1100}`, **clip outliers** clinical-bounds (Height 140–210cm, Weight 45–200kg), median impute.
- **Diet decoder**: BRFSS code XXX → per-day frequency (1XX/day, 2XX/week→/7, 3XX/month→/30, 555→0, 777/999→NaN).
- **Split**: default **temporal** (train 2019–2020, test 2021); `random` cho sensitivity check.
- **Class imbalance**: default `native_weight`; option SMOTE để bạn ablation sau (chưa default).
- **Survey weights** (`_LLCPWT`): flag `use_survey_weights` (default False).
- **Threshold**: tune trên val theo `threshold_objective` (F1/recall/precision).
- **Early stopping**: xgb + catboost (khi không HPO); tabtransformer theo val ROC-AUC.
- **HPO**: `use_hpo=True` → GridSearchCV với CV trên train cho logreg/xgb/catboost. TabTransformer không HPO (deep learning quá chậm cho grid search).
- **Reproducibility**: seed 42 đồng nhất.

## Reporting
Per-split prevalence, ROC-AUC, PR-AUC + **PR-AUC Lift** (PR_AUC / Prevalence), Brier, Accuracy/Precision/Recall/F1 ở 2 ngưỡng (0.5 và tuned), confusion matrix, calibration curve. Khi HPO bật → log `best_params`, `best_cv_score`.

## Flags chính trong CONFIG
| Flag | Giá trị | Ý nghĩa |
|---|---|---|
| `model_name` | `logreg`/`xgb`/`catboost`/`tabtransformer` | Model |
| `split_mode` | `temporal` (default) / `random` | Chiến lược chia |
| `temporal_test_year` | `2021` | Năm test |
| `imbalance_mode` | `native_weight` / `none` | Class imbalance |
| `preproc_variant` | `baseline`/`missing_indicator`/`explicit_unknown` | Biến thể prep |
| `clip_outliers` | `True` | Clip clinical bounds |
| `threshold_objective` | `f1`/`recall`/`precision` | Mục tiêu tune threshold |
| `use_survey_weights` | `False` | `_LLCPWT` |
| **`use_hpo`** | `False` | **GridSearchCV** |
| `hpo_cv_folds` | `3` | k-fold cho HPO |
| `hpo_scoring` | `roc_auc` | Mục tiêu HPO |

## Giới hạn & mở rộng
- HPO chỉ cho 3 model sklearn-compatible (logreg/xgb/catboost). TabTransformer cần manual tune.
- Chưa SMOTE/SMOTENC built-in (sẵn sàng làm ablation).
- Chưa LLM/generative paradigm.
- Gợi ý ablation: (a) `native_weight` vs `none`; (b) `temporal` vs `random`; (c) `use_survey_weights`; (d) `preproc_variant`; (e) `use_hpo`; (f) feature subset (15 cũ vs 18 mới).

In [ ]:
# =========================
# INSTALL DEPENDENCIES (Colab / fresh env)
# =========================
# Chạy cell này đầu tiên. Nếu package thiếu, cài vào đúng kernel (qua sys.executable).
# Lưu ý: sau lần cài đầu (đặc biệt catboost), phải "Runtime -> Restart session" rồi Run All lại.
import importlib, subprocess, sys

REQUIRED = [
    # (import_name, pip_spec)
    ("numpy",      "numpy"),
    ("pandas",     "pandas"),
    ("sklearn",    "scikit-learn"),
    ("xgboost",    "xgboost>=2.0"),   # early_stopping_rounds trong constructor cần 2.0+
    ("catboost",   "catboost"),        # Colab KHÔNG có sẵn
    ("matplotlib", "matplotlib"),
    ("torch",      "torch"),
]

missing_pkgs = []
for mod, pkg_spec in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing_pkgs.append(pkg_spec)

if missing_pkgs:
    print("Installing:", missing_pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])
    print("\n>>> DONE. Hãy RESTART RUNTIME (menu: Runtime -> Restart session) rồi Run All lại để load package mới.")
else:
    print("All required packages already installed. Tiếp tục chạy các cell bên dưới.")

In [ ]:
# =========================
# FLAGS / CONFIG
# =========================
from pathlib import Path

CONFIG = {
    # dữ liệu
    "y2019": "",   # ví dụ: "/content/LLCP2019_CSV.csv"
    "y2020": "",
    "y2021": "",

    # debug
    "max_rows_per_year": None,   # ví dụ: 20000 để smoke test
    "strict": False,

    # baseline choices
    "model_name": "xgb",         # "logreg" | "xgb" | "catboost" | "tabtransformer"
    "split_mode": "temporal",    # "temporal" (primary) | "random" (sensitivity)
    "temporal_test_year": 2021,
    "preproc_variant": "baseline",   # "baseline" | "missing_indicator" | "explicit_unknown"
    "imbalance_mode": "native_weight",  # "none" | "native_weight"
    "threshold_objective": "f1",  # "f1" | "recall" | "precision"

    # outlier filtering (clinical bounds, theo paper jcdd-11-00396)
    "clip_outliers": True,
    "height_min_cm": 140.0,
    "height_max_cm": 210.0,
    "weight_min_kg": 45.0,
    "weight_max_kg": 200.0,

    # survey weights
    "use_survey_weights": False,

    # evaluation
    "test_size": 0.20,
    "val_size_within_train": 0.20,
    "random_state": 42,

    # output
    "outdir": "./baseline_notebook_outputs",

    # HPO via GridSearchCV (chỉ áp dụng cho logreg/xgb/catboost)
    "use_hpo": False,
    "hpo_cv_folds": 3,
    "hpo_scoring": "roc_auc",
    "hpo_n_jobs": -1,

    # xgb early stopping (chỉ dùng khi use_hpo=False)
    "xgb_early_stopping_rounds": 20,

    # tabtransformer
    "tt_epochs": 25,
    "tt_batch_size": 1024,
    "tt_lr": 1e-3,
    "tt_weight_decay": 1e-5,
    "tt_d_token": 32,
    "tt_n_heads": 4,
    "tt_n_layers": 2,
    "tt_mlp_hidden": 64,
    "tt_dropout": 0.1,
    "tt_patience": 5,
}

print(CONFIG)

In [ ]:

# =========================
# IMPORTS
# =========================
import json
import math
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# =========================
# CONSTANTS / FEATURE MAP
# =========================
RANDOM_STATE = int(CONFIG["random_state"])
COMPLETED_CODES = {110, 1100}

FEATURE_CANDIDATES: Dict[str, List[str]] = {
    'General_Health':              ['GENHLTH'],
    'Checkup':                     ['CHECKUP1'],
    'Exercise':                    ['EXERANY2'],
    'Skin_Cancer':                 ['CHCSCNCR'],
    'Other_Cancer':                ['CHCOCNCR'],
    'Depression':                  ['ADDEPEV3'],
    'Diabetes':                    ['DIABETE4', 'DIABETE3'],
    'Arthritis':                   ['HAVARTH5', 'HAVARTH4'],
    'Sex':                         ['SEXVAR', '_SEX', 'BIRTHSEX'],
    'Age_Category':                ['_AGEG5YR', '_AGE65YR', '_AGE_G'],
    'Height_(cm)':                 ['HTM4', 'HTIN4', 'HEIGHT3'],
    'Weight_(kg)':                 ['WTKG3', 'WEIGHT2'],
    'BMI':                         ['_BMI5', 'BMI'],
    'Smoking_History':             ['SMOKE100'],
    'Alcohol_Consumption':         ['_DRNKWK1', 'AVEDRNK3', 'DRNKANY5'],
    # Diet features (theo paper jcdd-11-00396, raw frequency code XXX format)
    'Fruit_Consumption':           ['FRUIT2', 'FRUIT1'],
    'Green_Vegetables_Consumption':['FVGREEN1', 'FVGREEN'],
    'FriedPotato_Consumption':     ['FRENCHF1', 'FRENCHF'],
    'Heart_Disease':               ['_MICHD'],
}

INPUT_FEATURES = [
    'General_Health', 'Checkup', 'Exercise', 'Skin_Cancer', 'Other_Cancer',
    'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category',
    'Height_(cm)', 'Weight_(kg)', 'BMI', 'Smoking_History', 'Alcohol_Consumption',
    'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption',
]
TARGET_COL = 'Heart_Disease'
YEAR_COL = 'YEAR'
WEIGHT_COL = 'SurveyWeight'
RAW_WEIGHT_CANDIDATES = ['_LLCPWT']

# tabular typing
NOMINAL_COLS = ['Sex', 'Diabetes']
BINARY_COLS = ['Exercise', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Arthritis', 'Smoking_History']
ORDINAL_COLS = ['General_Health', 'Checkup', 'Age_Category']
NUMERIC_COLS = [
    'Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption',
    'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption',
]

TT_CATEGORICAL_COLS = BINARY_COLS + ORDINAL_COLS + NOMINAL_COLS
TT_CONTINUOUS_COLS = NUMERIC_COLS

SPECIAL_MISSING_GENERIC = {7, 8, 9, 77, 88, 99, 777, 888, 999, 7777, 8888, 9999, 99900}

# Grid search spaces (nhỏ gọn để chạy trong thời gian hợp lý trên ~500K rows)
HPO_GRIDS = {
    "logreg": {
        "C": [0.01, 0.1, 1.0, 10.0],
        "penalty": ["l2"],
        "solver": ["lbfgs"],
    },
    "xgb": {
        "max_depth": [3, 5, 7],
        "learning_rate": [0.05, 0.1],
        "n_estimators": [300, 500],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    },
    "catboost": {
        "depth": [4, 6, 8],
        "learning_rate": [0.05, 0.1],
        "iterations": [300, 500],
    },
}

@dataclass
class SplitData:
    X_train: pd.DataFrame
    X_val: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_val: pd.Series
    y_test: pd.Series
    train_years: List[int]
    test_years: List[int]
    w_train: Optional[pd.Series] = None
    w_val: Optional[pd.Series] = None
    w_test: Optional[pd.Series] = None

In [ ]:
# =========================
# DECODING HELPERS
# =========================
def _to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors='coerce')

def _replace_special_missing(series: pd.Series, extra_missing: Optional[Sequence[int]] = None) -> pd.Series:
    miss = set(SPECIAL_MISSING_GENERIC)
    if extra_missing is not None:
        miss.update(extra_missing)
    return series.replace(sorted(miss), np.nan)

def _decode_yes_no(series: pd.Series) -> pd.Series:
    s = _replace_special_missing(_to_numeric(series), extra_missing=[7, 9])
    return s.map({1: 1.0, 2: 0.0})

def _decode_target_from_michd(series: pd.Series) -> pd.Series:
    s = _replace_special_missing(_to_numeric(series), extra_missing=[7, 9])
    return s.map({1: 1.0, 2: 0.0})

def _decode_general_health(series: pd.Series) -> pd.Series:
    s = _replace_special_missing(_to_numeric(series), extra_missing=[7, 9])
    return s.map({1: 4.0, 2: 3.0, 3: 2.0, 4: 1.0, 5: 0.0})

def _decode_checkup(series: pd.Series) -> pd.Series:
    s = _to_numeric(series)
    s = s.replace([7, 9, 77, 99, 777, 999], np.nan)
    return s.map({1: 4.0, 2: 3.0, 3: 2.0, 4: 1.0, 8: 0.0})

def _decode_age_category(series: pd.Series, raw_name: str) -> pd.Series:
    s = _replace_special_missing(_to_numeric(series))
    if raw_name in {'_AGEG5YR', '_AGE65YR', '_AGE_G'}:
        return s - 1
    return s

def _decode_sex(series: pd.Series) -> pd.Series:
    s = _replace_special_missing(_to_numeric(series), extra_missing=[7, 9])
    return s.map({1: 'Male', 2: 'Female'})

def _decode_diabetes(series: pd.Series) -> pd.Series:
    s = _replace_special_missing(_to_numeric(series), extra_missing=[7, 9])
    return s.map({1: 'Yes', 2: 'Pregnancy', 3: 'No', 4: 'Prediabetes'})

def _height_to_cm(series: pd.Series, raw_name: str) -> pd.Series:
    s = _replace_special_missing(_to_numeric(series))
    if raw_name == 'HTM4':
        return s.astype(float)
    if raw_name == 'HTIN4':
        return s * 2.54
    if raw_name == 'HEIGHT3':
        feet = np.floor(s / 100)
        inches = s % 100
        out = (feet * 12 + inches) * 2.54
        out[(feet < 3) | (feet > 9)] = np.nan
        out[(inches < 0) | (inches > 11)] = np.nan
        return out
    return s.astype(float)

def _weight_to_kg(series: pd.Series, raw_name: str) -> pd.Series:
    s = _replace_special_missing(_to_numeric(series))
    if raw_name == 'WTKG3':
        return s / 100.0
    if raw_name == 'WEIGHT2':
        metric_mask = s >= 9000
        kg = s.copy().astype(float)
        kg[metric_mask] = s[metric_mask] - 9000
        kg[~metric_mask] = s[~metric_mask] / 2.2046
        return kg
    return s.astype(float)

def _bmi_to_numeric(series: pd.Series) -> pd.Series:
    s = _replace_special_missing(_to_numeric(series))
    return s / 100.0

def _weekly_drinks(series: pd.Series, raw_name: str) -> pd.Series:
    s = _to_numeric(series)
    if raw_name == '_DRNKWK1':
        return s.mask(s == 99900, np.nan).astype(float)
    if raw_name == 'AVEDRNK3':
        s = s.replace([77, 99], np.nan)
        return s.mask(s == 88, 0).astype(float)
    if raw_name == 'DRNKANY5':
        s = s.replace([7, 9], np.nan)
        return s.map({1: 1.0, 2: 0.0})
    return s.astype(float)

def _decode_food_frequency(series: pd.Series) -> pd.Series:
    """
    Decode BRFSS food-frequency codes (FRUIT2 / FVGREEN1 / FRENCHF1 / ...).
    Format BRFSS:
      101-199: X times per day  (e.g. 102 = 2/day)
      201-299: X times per week
      301-399: X times per month
      555    : Never
      777,999: Don't know / Refused -> NaN
    Output: số lần ăn quy đổi về ĐƠN VỊ "lần per day".
    """
    s = _to_numeric(series)
    out = pd.Series(np.nan, index=s.index, dtype=float)

    # Never -> 0
    out[s == 555] = 0.0

    # Per day
    mask_day = (s >= 101) & (s <= 199)
    out[mask_day] = (s[mask_day] - 100).astype(float)

    # Per week -> chia 7
    mask_week = (s >= 201) & (s <= 299)
    out[mask_week] = (s[mask_week] - 200).astype(float) / 7.0

    # Per month -> chia 30
    mask_month = (s >= 301) & (s <= 399)
    out[mask_month] = (s[mask_month] - 300).astype(float) / 30.0

    # 777, 999 (DK/Refused) đã là NaN do init; chỉ cần đảm bảo không leak qua các mask trên
    return out

In [ ]:
# =========================
# DATA LOADING / CANONICALIZATION
# =========================
def _read_header(path: Path) -> List[str]:
    cols = pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()
    return [c for c in cols if c != 'Unnamed: 0']

def _find_first_existing(columns: Iterable[str], candidates: Sequence[str]) -> Optional[str]:
    colset = set(columns)
    for cand in candidates:
        if cand in colset:
            return cand
    return None

def resolve_mapping(columns: Sequence[str]) -> Dict[str, Optional[str]]:
    return {feat: _find_first_existing(columns, candidates) for feat, candidates in FEATURE_CANDIDATES.items()}

def required_usecols(columns: Sequence[str]) -> List[str]:
    required = {'DISPCODE', 'CVDINFR4', 'CVDCRHD4'} | set(RAW_WEIGHT_CANDIDATES)
    for candidates in FEATURE_CANDIDATES.values():
        required.update(candidates)
    return [c for c in columns if c in required and c != 'Unnamed: 0']

def load_year_raw(path: Path, max_rows: Optional[int] = None):
    columns = _read_header(path)
    usecols = required_usecols(columns)
    df = pd.read_csv(path, usecols=usecols, nrows=max_rows, low_memory=False)
    mapping = resolve_mapping(df.columns.tolist())
    return df, mapping

def _apply_outlier_clip(series: pd.Series, lo: float, hi: float) -> pd.Series:
    s = series.copy()
    s = s.where((s >= lo) & (s <= hi), np.nan)
    return s

def build_year_dataframe(path: Path, year: int, max_rows: Optional[int] = None, strict: bool = False):
    raw_df, mapping = load_year_raw(path, max_rows=max_rows)
    missing = [k for k, v in mapping.items() if v is None and k != TARGET_COL]
    if strict and missing:
        raise ValueError(f"Year {year}: missing mappings for {missing}")

    n_loaded = len(raw_df)
    if 'DISPCODE' in raw_df.columns:
        raw_df = raw_df[raw_df['DISPCODE'].isin(COMPLETED_CODES)].copy()
    n_completed = len(raw_df)

    out = pd.DataFrame(index=raw_df.index)

    target_raw = mapping.get(TARGET_COL)
    if target_raw is not None:
        out[TARGET_COL] = _decode_target_from_michd(raw_df[target_raw])
    elif all(c in raw_df.columns for c in ['CVDINFR4', 'CVDCRHD4']):
        inf = _replace_special_missing(_to_numeric(raw_df['CVDINFR4']), extra_missing=[7, 9])
        chd = _replace_special_missing(_to_numeric(raw_df['CVDCRHD4']), extra_missing=[7, 9])
        target = pd.Series(np.nan, index=raw_df.index, dtype=float)
        target[(inf == 1) | (chd == 1)] = 1.0
        target[(inf == 2) & (chd == 2)] = 0.0
        out[TARGET_COL] = target
    else:
        raise ValueError(f"Year {year}: cannot build target")

    if mapping['General_Health']: out['General_Health'] = _decode_general_health(raw_df[mapping['General_Health']])
    if mapping['Checkup']: out['Checkup'] = _decode_checkup(raw_df[mapping['Checkup']])
    if mapping['Exercise']: out['Exercise'] = _decode_yes_no(raw_df[mapping['Exercise']])
    if mapping['Skin_Cancer']: out['Skin_Cancer'] = _decode_yes_no(raw_df[mapping['Skin_Cancer']])
    if mapping['Other_Cancer']: out['Other_Cancer'] = _decode_yes_no(raw_df[mapping['Other_Cancer']])
    if mapping['Depression']: out['Depression'] = _decode_yes_no(raw_df[mapping['Depression']])
    if mapping['Diabetes']: out['Diabetes'] = _decode_diabetes(raw_df[mapping['Diabetes']])
    if mapping['Arthritis']: out['Arthritis'] = _decode_yes_no(raw_df[mapping['Arthritis']])
    if mapping['Sex']: out['Sex'] = _decode_sex(raw_df[mapping['Sex']])
    if mapping['Age_Category']: out['Age_Category'] = _decode_age_category(raw_df[mapping['Age_Category']], mapping['Age_Category'])
    if mapping['Height_(cm)']: out['Height_(cm)'] = _height_to_cm(raw_df[mapping['Height_(cm)']], mapping['Height_(cm)'])
    if mapping['Weight_(kg)']: out['Weight_(kg)'] = _weight_to_kg(raw_df[mapping['Weight_(kg)']], mapping['Weight_(kg)'])
    if mapping['BMI']: out['BMI'] = _bmi_to_numeric(raw_df[mapping['BMI']])
    if mapping['Smoking_History']: out['Smoking_History'] = _decode_yes_no(raw_df[mapping['Smoking_History']])
    if mapping['Alcohol_Consumption']: out['Alcohol_Consumption'] = _weekly_drinks(raw_df[mapping['Alcohol_Consumption']], mapping['Alcohol_Consumption'])

    # Diet features
    if mapping.get('Fruit_Consumption'):
        out['Fruit_Consumption'] = _decode_food_frequency(raw_df[mapping['Fruit_Consumption']])
    if mapping.get('Green_Vegetables_Consumption'):
        out['Green_Vegetables_Consumption'] = _decode_food_frequency(raw_df[mapping['Green_Vegetables_Consumption']])
    if mapping.get('FriedPotato_Consumption'):
        out['FriedPotato_Consumption'] = _decode_food_frequency(raw_df[mapping['FriedPotato_Consumption']])

    # Outlier clipping (clinical bounds) cho height/weight, theo paper jcdd-11-00396
    if CONFIG.get("clip_outliers", True):
        if 'Height_(cm)' in out.columns:
            out['Height_(cm)'] = _apply_outlier_clip(
                out['Height_(cm)'], CONFIG["height_min_cm"], CONFIG["height_max_cm"]
            )
        if 'Weight_(kg)' in out.columns:
            out['Weight_(kg)'] = _apply_outlier_clip(
                out['Weight_(kg)'], CONFIG["weight_min_kg"], CONFIG["weight_max_kg"]
            )

    # Survey weight
    weight_raw = _find_first_existing(raw_df.columns.tolist(), RAW_WEIGHT_CANDIDATES)
    if weight_raw is not None:
        out[WEIGHT_COL] = _to_numeric(raw_df[weight_raw]).astype(float)
    else:
        out[WEIGHT_COL] = np.nan

    # Drop rows missing target
    n_before_target_drop = len(out)
    out = out.dropna(subset=[TARGET_COL]).copy()
    n_after_target_drop = len(out)
    n_dropped_target_na = n_before_target_drop - n_after_target_drop
    out[YEAR_COL] = year

    info = {
        "year": year,
        "n_loaded": int(n_loaded),
        "n_completed_interview": int(n_completed),
        "n_dropped_target_na": int(n_dropped_target_na),
        "n_target_valid": int(n_after_target_drop),
        "positive_rate": float(out[TARGET_COL].mean()) if len(out) else None,
        "resolved_mapping": mapping,
        "weight_var": weight_raw,
        "n_weight_present": int(out[WEIGHT_COL].notna().sum()),
    }
    return out, info

def merge_years(year_to_path: Mapping[int, str], max_rows: Optional[int] = None, strict: bool = False):
    frames, infos = [], {}
    for year, raw_path in sorted(year_to_path.items()):
        df_year, info = build_year_dataframe(Path(raw_path), year, max_rows=max_rows, strict=strict)
        frames.append(df_year)
        infos[year] = info
    merged = pd.concat(frames, axis=0, ignore_index=True)
    return merged, infos

year_to_path = {year: CONFIG[f"y{year}"] for year in [2019, 2020, 2021] if str(CONFIG[f"y{year}"]).strip()}
if not year_to_path:
    raise ValueError("Hãy điền ít nhất một path dữ liệu vào CONFIG.")
for y, p in year_to_path.items():
    if not Path(p).exists():
        raise FileNotFoundError(f"Không tìm thấy file năm {y}: {p}")

df_all, year_infos = merge_years(
    year_to_path=year_to_path,
    max_rows=CONFIG["max_rows_per_year"],
    strict=CONFIG["strict"]
)

print("Rows:", len(df_all))
print("Years:", sorted(df_all[YEAR_COL].unique().tolist()))
print("Positive rate:", round(float(df_all[TARGET_COL].mean()), 4))
print("\nRow attrition per year:")
for y, info in sorted(year_infos.items()):
    print(f"  {y}: loaded={info['n_loaded']:>7} | completed={info['n_completed_interview']:>7} "
          f"| dropped_target_na={info['n_dropped_target_na']:>6} | kept={info['n_target_valid']:>7} "
          f"| pos_rate={info['positive_rate']:.4f} | weight_var={info['weight_var']}")

display(df_all.head())
display(pd.DataFrame(year_infos).T)

In [ ]:
# =========================
# ENFORCE FEATURE INTERSECTION ACROSS YEARS
# =========================
# Yêu cầu: input features phải available ở TẤT CẢ các năm được load.
# Nếu một canonical feature không resolve được ở bất kỳ năm nào -> loại luôn.
# Target (Heart_Disease) được xử lý riêng trong build_year_dataframe
# (có fallback từ CVDINFR4/CVDCRHD4) nên không dùng trong intersection check.

resolved_by_year = {
    y: {
        feat for feat, raw in info["resolved_mapping"].items()
        if raw is not None and feat != TARGET_COL
    }
    for y, info in year_infos.items()
}

intersect_features = (
    set.intersection(*resolved_by_year.values()) if resolved_by_year else set()
)

kept_input_features = [f for f in INPUT_FEATURES if f in intersect_features]
dropped_input_features = [f for f in INPUT_FEATURES if f not in intersect_features]

print("=== Resolved canonical features per year ===")
for y, feats in sorted(resolved_by_year.items()):
    missing_here = sorted(set(INPUT_FEATURES) - feats)
    print(f"  {y}: n_resolved={len(feats)} | missing_from_INPUT={missing_here}")

print(f"\nIntersection across years ({len(kept_input_features)}): {kept_input_features}")
print(f"Dropped (not in every year) ({len(dropped_input_features)}): {dropped_input_features}")

# Rebind canonical feature groups to the intersection set
INPUT_FEATURES = kept_input_features
NOMINAL_COLS = [c for c in NOMINAL_COLS if c in intersect_features]
BINARY_COLS = [c for c in BINARY_COLS if c in intersect_features]
ORDINAL_COLS = [c for c in ORDINAL_COLS if c in intersect_features]
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in intersect_features]
TT_CATEGORICAL_COLS = BINARY_COLS + ORDINAL_COLS + NOMINAL_COLS
TT_CONTINUOUS_COLS = NUMERIC_COLS

# Trim df_all to intersection-only feature columns (plus target, year, survey weight)
keep_cols = INPUT_FEATURES + [TARGET_COL, YEAR_COL, WEIGHT_COL]
df_all = df_all[keep_cols].copy()

print(f"\nFinal df_all shape: {df_all.shape}")
print(f"Final INPUT_FEATURES ({len(INPUT_FEATURES)}): {INPUT_FEATURES}")
print(f"  NOMINAL_COLS: {NOMINAL_COLS}")
print(f"  BINARY_COLS:  {BINARY_COLS}")
print(f"  ORDINAL_COLS: {ORDINAL_COLS}")
print(f"  NUMERIC_COLS: {NUMERIC_COLS}")
print(f"  {WEIGHT_COL} non-null: {int(df_all[WEIGHT_COL].notna().sum())} / {len(df_all)}")

if len(INPUT_FEATURES) == 0:
    raise ValueError(
        "Intersection rỗng: không có feature nào available ở tất cả các năm. "
        "Kiểm tra lại FEATURE_CANDIDATES hoặc danh sách năm."
    )


In [ ]:
# =========================
# SPLIT
# =========================
def _build_weights(df_slice: pd.DataFrame) -> Optional[pd.Series]:
    """Return weight series if CONFIG['use_survey_weights']=True, else None.
    Rows with missing weight fall back to 1.0 so training never crashes."""
    if not CONFIG.get("use_survey_weights", False):
        return None
    w = df_slice[WEIGHT_COL].astype(float).copy()
    w = w.fillna(1.0)
    w[w <= 0] = 1.0
    return w.reset_index(drop=True)

def make_split_random(df: pd.DataFrame) -> SplitData:
    X = df[INPUT_FEATURES + [YEAR_COL, WEIGHT_COL]].copy()
    y = df[TARGET_COL].astype(int).copy()
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y,
        test_size=float(CONFIG["test_size"]),
        stratify=y,
        random_state=RANDOM_STATE,
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size=float(CONFIG["val_size_within_train"]),
        stratify=y_trainval,
        random_state=RANDOM_STATE,
    )

    w_train = _build_weights(X_train)
    w_val = _build_weights(X_val)
    w_test = _build_weights(X_test)

    return SplitData(
        X_train=X_train.reset_index(drop=True),
        X_val=X_val.reset_index(drop=True),
        X_test=X_test.reset_index(drop=True),
        y_train=y_train.reset_index(drop=True),
        y_val=y_val.reset_index(drop=True),
        y_test=y_test.reset_index(drop=True),
        train_years=sorted(X_train[YEAR_COL].unique().tolist()),
        test_years=sorted(X_test[YEAR_COL].unique().tolist()),
        w_train=w_train, w_val=w_val, w_test=w_test,
    )

def make_split_temporal(df: pd.DataFrame, test_year: int) -> SplitData:
    trainval_df = df[df[YEAR_COL] < test_year].copy()
    test_df = df[df[YEAR_COL] == test_year].copy()
    if len(trainval_df) == 0 or len(test_df) == 0:
        raise ValueError(
            f"Temporal split cần có ít nhất một năm train < {test_year} và test year = {test_year}."
        )
    X_trainval = trainval_df[INPUT_FEATURES + [YEAR_COL, WEIGHT_COL]].copy()
    y_trainval = trainval_df[TARGET_COL].astype(int).copy()
    X_test = test_df[INPUT_FEATURES + [YEAR_COL, WEIGHT_COL]].copy().reset_index(drop=True)
    y_test = test_df[TARGET_COL].astype(int).copy().reset_index(drop=True)
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size=float(CONFIG["val_size_within_train"]),
        stratify=y_trainval,
        random_state=RANDOM_STATE,
    )

    w_train = _build_weights(X_train)
    w_val = _build_weights(X_val)
    w_test = _build_weights(X_test)

    return SplitData(
        X_train=X_train.reset_index(drop=True),
        X_val=X_val.reset_index(drop=True),
        X_test=X_test,
        y_train=y_train.reset_index(drop=True),
        y_val=y_val.reset_index(drop=True),
        y_test=y_test,
        train_years=sorted(X_train[YEAR_COL].unique().tolist()),
        test_years=sorted(X_test[YEAR_COL].unique().tolist()),
        w_train=w_train, w_val=w_val, w_test=w_test,
    )

if CONFIG["split_mode"] == "random":
    split_data = make_split_random(df_all)
else:
    split_data = make_split_temporal(df_all, test_year=int(CONFIG["temporal_test_year"]))

print("Split mode:", CONFIG["split_mode"])
print("Train/Val/Test sizes:", len(split_data.X_train), len(split_data.X_val), len(split_data.X_test))
print("Train years:", split_data.train_years, "| Test years:", split_data.test_years)
print(f"Positive rate  train={split_data.y_train.mean():.4f}  "
      f"val={split_data.y_val.mean():.4f}  test={split_data.y_test.mean():.4f}")
if split_data.w_train is not None:
    print(f"Survey weights  train_sum={float(split_data.w_train.sum()):.1f}  "
          f"val_sum={float(split_data.w_val.sum()):.1f}  test_sum={float(split_data.w_test.sum()):.1f}")
else:
    print("Survey weights: DISABLED (use_survey_weights=False)")

In [ ]:
# =========================
# METRICS / EVAL HELPERS
# =========================
def _to_np_weights(w):
    return None if w is None else np.asarray(w, dtype=float)

def compute_metrics(y_true: np.ndarray, y_score: np.ndarray, threshold: float,
                    sample_weight: Optional[np.ndarray] = None):
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    sw = _to_np_weights(sample_weight)
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred, sample_weight=sw)),
        "Precision": float(precision_score(y_true, y_pred, sample_weight=sw, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, sample_weight=sw, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, sample_weight=sw, zero_division=0)),
        "Threshold": float(threshold),
        "ConfusionMatrix": confusion_matrix(y_true, y_pred, sample_weight=sw),
    }

def compute_prob_metrics(y_true: np.ndarray, y_score: np.ndarray,
                         sample_weight: Optional[np.ndarray] = None):
    sw = _to_np_weights(sample_weight)
    y_true_arr = np.asarray(y_true)
    prevalence = (
        float(np.average(y_true_arr, weights=sw)) if sw is not None
        else float(y_true_arr.mean())
    )
    pr_auc = float(average_precision_score(y_true_arr, y_score, sample_weight=sw))
    # lift vs constant predictor (PR-AUC của random = prevalence)
    pr_auc_lift = (pr_auc / prevalence) if prevalence > 0 else float('nan')
    return {
        "ROC_AUC": float(roc_auc_score(y_true_arr, y_score, sample_weight=sw)),
        "PR_AUC": pr_auc,
        "Prevalence": prevalence,
        "PR_AUC_Lift": pr_auc_lift,
        "Brier": float(brier_score_loss(y_true_arr, y_score, sample_weight=sw)),
    }

def find_best_threshold(y_true: np.ndarray, y_score: np.ndarray, objective: str = "f1",
                        sample_weight: Optional[np.ndarray] = None):
    thresholds = np.linspace(0.05, 0.95, 181)
    sw = _to_np_weights(sample_weight)
    best_thr, best_value, best_metrics = 0.5, -1.0, {}
    for thr in thresholds:
        y_pred = (y_score >= thr).astype(int)
        precision = precision_score(y_true, y_pred, sample_weight=sw, zero_division=0)
        recall = recall_score(y_true, y_pred, sample_weight=sw, zero_division=0)
        f1 = f1_score(y_true, y_pred, sample_weight=sw, zero_division=0)
        score = recall if objective == "recall" else precision if objective == "precision" else f1
        if score > best_value:
            best_value = score
            best_thr = float(thr)
            best_metrics = {"Precision": float(precision), "Recall": float(recall), "F1": float(f1)}
    return best_thr, best_metrics

def show_confusion_matrix(cm, title="Confusion Matrix"):
    _, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:.0f}", ha="center", va="center")
    plt.show()

def show_calibration(y_true, y_score, title="Calibration"):
    prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=10, strategy='quantile')
    plt.figure(figsize=(5, 4))
    plt.plot(prob_pred, prob_true, marker='o')
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.title(title)
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed positive rate")
    plt.show()


In [ ]:

# =========================
# MATRIX PREPROCESSOR FOR LOGREG / XGB
# =========================
def class_ratio(y: Sequence[int]) -> float:
    y_arr = np.asarray(y)
    pos = int((y_arr == 1).sum())
    neg = int((y_arr == 0).sum())
    return float(neg / max(pos, 1))

def make_matrix_preprocessor(model_name: str, preproc_variant: str):
    numeric_like = [c for c in (BINARY_COLS + ORDINAL_COLS + NUMERIC_COLS) if c in INPUT_FEATURES]
    nominal = [c for c in NOMINAL_COLS if c in INPUT_FEATURES]
    scale_numeric = model_name == "logreg"

    numeric_steps = []
    if preproc_variant == "missing_indicator":
        numeric_steps.append(("imputer", SimpleImputer(strategy="median", add_indicator=True)))
    else:
        numeric_steps.append(("imputer", SimpleImputer(strategy="median")))
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler(with_mean=False)))

    cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown") if preproc_variant == "explicit_unknown" else SimpleImputer(strategy="most_frequent")

    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric_like),
            ("categorical", Pipeline([
                ("imputer", cat_imputer),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), nominal),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

In [ ]:
# =========================
# MODEL RUNNERS: LOGREG / XGB / CATBOOST  (+ optional GridSearchCV HPO)
# =========================
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def class_ratio_weighted(y: Sequence[int], w: Optional[Sequence[float]] = None) -> float:
    y_arr = np.asarray(y)
    if w is None:
        pos = float((y_arr == 1).sum())
        neg = float((y_arr == 0).sum())
    else:
        w_arr = np.asarray(w, dtype=float)
        pos = float(w_arr[y_arr == 1].sum())
        neg = float(w_arr[y_arr == 0].sum())
    return float(neg / max(pos, 1e-12))

def _make_cv():
    return StratifiedKFold(
        n_splits=int(CONFIG["hpo_cv_folds"]),
        shuffle=True,
        random_state=RANDOM_STATE,
    )

def _gridsearch_fit(base_model, grid, X, y, sample_weight=None):
    cv = _make_cv()
    search = GridSearchCV(
        estimator=base_model,
        param_grid=grid,
        scoring=CONFIG["hpo_scoring"],
        cv=cv,
        n_jobs=int(CONFIG["hpo_n_jobs"]),
        refit=True,
        verbose=1,
    )
    fit_kwargs = {} if sample_weight is None else {"sample_weight": sample_weight}
    search.fit(X, y, **fit_kwargs)
    print(f"  [HPO] best score = {search.best_score_:.4f} | params = {search.best_params_}")
    return search.best_estimator_, search.best_params_, float(search.best_score_)

def run_logreg_or_xgb(model_name: str, split_data: SplitData):
    pre = make_matrix_preprocessor(model_name, CONFIG["preproc_variant"])

    X_train = split_data.X_train[INPUT_FEATURES].copy()
    X_val = split_data.X_val[INPUT_FEATURES].copy()
    X_test = split_data.X_test[INPUT_FEATURES].copy()

    Xt_train = pre.fit_transform(X_train)
    Xt_val = pre.transform(X_val)
    Xt_test = pre.transform(X_test)

    w_train_np = None if split_data.w_train is None else split_data.w_train.to_numpy(dtype=float)
    w_val_np = None if split_data.w_val is None else split_data.w_val.to_numpy(dtype=float)
    w_test_np = None if split_data.w_test is None else split_data.w_test.to_numpy(dtype=float)

    ratio = class_ratio_weighted(split_data.y_train, w_train_np)
    use_hpo = bool(CONFIG.get("use_hpo", False))

    best_params = None
    best_cv_score = None
    best_iter = None

    if model_name == "logreg":
        if use_hpo:
            base = LogisticRegression(
                max_iter=4000,
                class_weight="balanced" if CONFIG["imbalance_mode"] == "native_weight" else None,
                random_state=RANDOM_STATE,
            )
            model, best_params, best_cv_score = _gridsearch_fit(
                base, HPO_GRIDS["logreg"], Xt_train, split_data.y_train, sample_weight=w_train_np
            )
        else:
            model = LogisticRegression(
                max_iter=4000,
                class_weight="balanced" if CONFIG["imbalance_mode"] == "native_weight" else None,
                random_state=RANDOM_STATE,
            )
            model.fit(Xt_train, split_data.y_train, sample_weight=w_train_np)

    elif model_name == "xgb":
        if use_hpo:
            # Khi HPO: KHÔNG dùng early_stopping (n_estimators trong grid)
            base = XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                tree_method="hist",
                n_jobs=1,
                reg_lambda=1.0,
                scale_pos_weight=(ratio if CONFIG["imbalance_mode"] == "native_weight" else 1.0),
            )
            model, best_params, best_cv_score = _gridsearch_fit(
                base, HPO_GRIDS["xgb"], Xt_train, split_data.y_train, sample_weight=w_train_np
            )
        else:
            model = XGBClassifier(
                n_estimators=1000,
                learning_rate=0.05,
                max_depth=5,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                tree_method="hist",
                n_jobs=1,
                scale_pos_weight=(ratio if CONFIG["imbalance_mode"] == "native_weight" else 1.0),
                early_stopping_rounds=int(CONFIG["xgb_early_stopping_rounds"]),
            )
            model.fit(
                Xt_train, split_data.y_train,
                sample_weight=w_train_np,
                eval_set=[(Xt_val, split_data.y_val)],
                sample_weight_eval_set=None if w_val_np is None else [w_val_np],
                verbose=False,
            )
            best_iter = int(getattr(model, "best_iteration", -1) or -1)
    else:
        raise ValueError(model_name)

    val_score = model.predict_proba(Xt_val)[:, 1]
    test_score = model.predict_proba(Xt_test)[:, 1]

    y_val_np = split_data.y_val.to_numpy()
    y_test_np = split_data.y_test.to_numpy()

    tuned_thr, tuned_metrics_val = find_best_threshold(
        y_val_np, np.asarray(val_score),
        CONFIG["threshold_objective"], sample_weight=w_val_np,
    )
    metrics_05 = compute_metrics(y_test_np, np.asarray(test_score), 0.5, sample_weight=w_test_np)
    metrics_tuned = compute_metrics(y_test_np, np.asarray(test_score), tuned_thr, sample_weight=w_test_np)
    prob_metrics = compute_prob_metrics(y_test_np, np.asarray(test_score), sample_weight=w_test_np)

    return {
        "model_name": model_name,
        "training_paradigm": "supervised",
        "do_hpo": use_hpo,
        "best_params": best_params,
        "best_cv_score": best_cv_score,
        "best_iteration": best_iter,
        "metrics_0.5": metrics_05,
        "metrics_tuned": metrics_tuned,
        "prob_metrics": prob_metrics,
        "tuned_threshold": tuned_thr,
        "val_best_metrics_at_tuned_threshold": tuned_metrics_val,
        "y_score_test": np.asarray(test_score),
    }

def prepare_catboost_frame(X: pd.DataFrame, preproc_variant: str):
    out = X.copy()
    for col in NOMINAL_COLS:
        if col in out.columns:
            out[col] = out[col].fillna("Unknown").astype(str)
    for col in BINARY_COLS + ORDINAL_COLS + NUMERIC_COLS:
        if col in out.columns:
            if preproc_variant == "missing_indicator":
                out[f"{col}__missing"] = out[col].isna().astype(int)
                out[col] = out[col].fillna(out[col].median())
            elif preproc_variant == "explicit_unknown":
                out[col] = out[col].fillna(out[col].median())
    return out

def run_catboost(split_data: SplitData):
    X_train = prepare_catboost_frame(split_data.X_train[INPUT_FEATURES].copy(), CONFIG["preproc_variant"])
    X_val = prepare_catboost_frame(split_data.X_val[INPUT_FEATURES].copy(), CONFIG["preproc_variant"])
    X_test = prepare_catboost_frame(split_data.X_test[INPUT_FEATURES].copy(), CONFIG["preproc_variant"])

    cat_features_idx = [X_train.columns.get_loc(c) for c in X_train.columns if c in NOMINAL_COLS]

    w_train_np = None if split_data.w_train is None else split_data.w_train.to_numpy(dtype=float)
    w_val_np = None if split_data.w_val is None else split_data.w_val.to_numpy(dtype=float)
    w_test_np = None if split_data.w_test is None else split_data.w_test.to_numpy(dtype=float)

    ratio = class_ratio_weighted(split_data.y_train, w_train_np)
    class_weights = [1.0, ratio] if CONFIG["imbalance_mode"] == "native_weight" else None
    use_hpo = bool(CONFIG.get("use_hpo", False))

    best_params = None
    best_cv_score = None
    best_iter = None

    if use_hpo:
        # CatBoost sklearn-compatible: cat_features qua constructor
        base = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=RANDOM_STATE,
            verbose=False,
            class_weights=class_weights,
            thread_count=1,
            cat_features=cat_features_idx,
        )
        model, best_params, best_cv_score = _gridsearch_fit(
            base, HPO_GRIDS["catboost"], X_train, split_data.y_train, sample_weight=w_train_np
        )
    else:
        model = CatBoostClassifier(
            iterations=1000,
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=RANDOM_STATE,
            verbose=False,
            class_weights=class_weights,
            thread_count=1,
            early_stopping_rounds=20,
        )
        train_pool = Pool(X_train, label=split_data.y_train, weight=w_train_np, cat_features=cat_features_idx)
        val_pool = Pool(X_val, label=split_data.y_val, weight=w_val_np, cat_features=cat_features_idx)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)
        best_iter = int(model.get_best_iteration() or -1)

    val_score = model.predict_proba(X_val)[:, 1]
    test_score = model.predict_proba(X_test)[:, 1]

    y_val_np = split_data.y_val.to_numpy()
    y_test_np = split_data.y_test.to_numpy()

    tuned_thr, tuned_metrics_val = find_best_threshold(
        y_val_np, np.asarray(val_score),
        CONFIG["threshold_objective"], sample_weight=w_val_np,
    )
    metrics_05 = compute_metrics(y_test_np, np.asarray(test_score), 0.5, sample_weight=w_test_np)
    metrics_tuned = compute_metrics(y_test_np, np.asarray(test_score), tuned_thr, sample_weight=w_test_np)
    prob_metrics = compute_prob_metrics(y_test_np, np.asarray(test_score), sample_weight=w_test_np)

    return {
        "model_name": "catboost",
        "training_paradigm": "supervised",
        "do_hpo": use_hpo,
        "best_params": best_params,
        "best_cv_score": best_cv_score,
        "best_iteration": best_iter,
        "metrics_0.5": metrics_05,
        "metrics_tuned": metrics_tuned,
        "prob_metrics": prob_metrics,
        "tuned_threshold": tuned_thr,
        "val_best_metrics_at_tuned_threshold": tuned_metrics_val,
        "y_score_test": np.asarray(test_score),
    }

In [ ]:
# =========================
# TABTRANSFORMER PREP
# =========================
def fit_tabtransformer_processors(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    X_train = train_df.copy()
    X_val = val_df.copy()
    X_test = test_df.copy()

    cat_maps = {}
    cat_cardinalities = []

    for col in TT_CATEGORICAL_COLS:
        train_col = X_train[col].copy()
        val_col = X_val[col].copy()
        test_col = X_test[col].copy()

        # cast to string tokens; keep explicit missing token
        train_col = train_col.astype("object").where(~train_col.isna(), "__MISSING__").astype(str)
        val_col = val_col.astype("object").where(~val_col.isna(), "__MISSING__").astype(str)
        test_col = test_col.astype("object").where(~test_col.isna(), "__MISSING__").astype(str)

        # Vocab: chỉ fit từ TRAIN (+ "__UNK__" cho giá trị chưa từng thấy ở val/test)
        train_tokens = sorted(set(train_col.unique().tolist()) | {"__UNK__"})
        stoi = {tok: i for i, tok in enumerate(train_tokens)}
        cat_maps[col] = stoi
        cat_cardinalities.append(len(train_tokens))

        unk_idx = stoi["__UNK__"]
        X_train[col] = train_col.map(stoi).fillna(unk_idx).astype(int)
        X_val[col] = val_col.map(stoi).fillna(unk_idx).astype(int)
        X_test[col] = test_col.map(stoi).fillna(unk_idx).astype(int)

    cont_stats = {}
    for col in TT_CONTINUOUS_COLS:
        median = float(X_train[col].median()) if not np.isnan(X_train[col].median()) else 0.0
        X_train[col] = X_train[col].fillna(median)
        X_val[col] = X_val[col].fillna(median)
        X_test[col] = X_test[col].fillna(median)

        mean = float(X_train[col].mean())
        std = float(X_train[col].std()) if float(X_train[col].std()) > 0 else 1.0
        cont_stats[col] = {"median": median, "mean": mean, "std": std}

        X_train[col] = (X_train[col] - mean) / std
        X_val[col] = (X_val[col] - mean) / std
        X_test[col] = (X_test[col] - mean) / std

    return X_train, X_val, X_test, cat_cardinalities

class TabularTorchDataset(Dataset):
    def __init__(self, X_cat: np.ndarray, X_cont: np.ndarray, y: np.ndarray,
                 w: Optional[np.ndarray] = None):
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.X_cont = torch.tensor(X_cont, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.w = (torch.tensor(w, dtype=torch.float32)
                  if w is not None
                  else torch.ones_like(self.y))
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_cont[idx], self.y[idx], self.w[idx]

class TabTransformerModel(nn.Module):
    def __init__(self, cat_cardinalities, n_cont, d_token=32, n_heads=4, n_layers=2, mlp_hidden=64, dropout=0.1):
        super().__init__()
        self.cat_embeddings = nn.ModuleList([nn.Embedding(card, d_token) for card in cat_cardinalities])
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.cont_proj = nn.Linear(n_cont, d_token)
        self.head = nn.Sequential(
            nn.Linear(d_token * (len(cat_cardinalities) + 1), mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

    def forward(self, x_cat, x_cont):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)]
        x = torch.stack(embs, dim=1)  # [B, n_cat, d_token]
        x = self.transformer(x)
        cont_tok = self.cont_proj(x_cont).unsqueeze(1)  # [B, 1, d_token]
        x = torch.cat([x, cont_tok], dim=1)
        x = x.reshape(x.size(0), -1)
        return self.head(x).squeeze(1)

def run_tabtransformer(split_data: SplitData):
    # Reproducibility
    torch.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    X_train = split_data.X_train[INPUT_FEATURES].copy()
    X_val = split_data.X_val[INPUT_FEATURES].copy()
    X_test = split_data.X_test[INPUT_FEATURES].copy()

    X_train, X_val, X_test, cat_cardinalities = fit_tabtransformer_processors(X_train, X_val, X_test)

    w_train_np = None if split_data.w_train is None else split_data.w_train.to_numpy(dtype=np.float32)
    w_val_np = None if split_data.w_val is None else split_data.w_val.to_numpy(dtype=np.float32)
    w_test_np = None if split_data.w_test is None else split_data.w_test.to_numpy(dtype=np.float32)

    tr_ds = TabularTorchDataset(
        X_train[TT_CATEGORICAL_COLS].to_numpy(),
        X_train[TT_CONTINUOUS_COLS].to_numpy(dtype=np.float32),
        split_data.y_train.to_numpy(dtype=np.float32),
        w_train_np,
    )
    va_ds = TabularTorchDataset(
        X_val[TT_CATEGORICAL_COLS].to_numpy(),
        X_val[TT_CONTINUOUS_COLS].to_numpy(dtype=np.float32),
        split_data.y_val.to_numpy(dtype=np.float32),
        w_val_np,
    )
    te_ds = TabularTorchDataset(
        X_test[TT_CATEGORICAL_COLS].to_numpy(),
        X_test[TT_CONTINUOUS_COLS].to_numpy(dtype=np.float32),
        split_data.y_test.to_numpy(dtype=np.float32),
        w_test_np,
    )

    g = torch.Generator(); g.manual_seed(RANDOM_STATE)
    tr_loader = DataLoader(tr_ds, batch_size=int(CONFIG["tt_batch_size"]), shuffle=True, generator=g)
    va_loader = DataLoader(va_ds, batch_size=int(CONFIG["tt_batch_size"]), shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=int(CONFIG["tt_batch_size"]), shuffle=False)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = TabTransformerModel(
        cat_cardinalities=cat_cardinalities,
        n_cont=len(TT_CONTINUOUS_COLS),
        d_token=int(CONFIG["tt_d_token"]),
        n_heads=int(CONFIG["tt_n_heads"]),
        n_layers=int(CONFIG["tt_n_layers"]),
        mlp_hidden=int(CONFIG["tt_mlp_hidden"]),
        dropout=float(CONFIG["tt_dropout"]),
    ).to(device)

    # pos_weight từ class imbalance (tính trên weighted counts nếu có survey weight)
    ratio = class_ratio_weighted(split_data.y_train, w_train_np)
    pos_weight = (
        torch.tensor([ratio], dtype=torch.float32, device=device)
        if CONFIG["imbalance_mode"] == "native_weight" else None
    )
    # Dùng reduction="none" để có thể nhân sample_weight per-row
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(CONFIG["tt_lr"]),
        weight_decay=float(CONFIG["tt_weight_decay"]),
    )

    best_state = None
    best_val_auc = -1.0
    patience = int(CONFIG["tt_patience"])
    patience_count = 0

    def predict_scores(loader):
        model.eval()
        ys, scores, ws = [], [], []
        with torch.no_grad():
            for xb_cat, xb_cont, yb, wb in loader:
                xb_cat, xb_cont = xb_cat.to(device), xb_cont.to(device)
                logits = model(xb_cat, xb_cont)
                probs = torch.sigmoid(logits).cpu().numpy()
                scores.append(probs)
                ys.append(yb.numpy())
                ws.append(wb.numpy())
        return np.concatenate(ys), np.concatenate(scores), np.concatenate(ws)

    for epoch in range(int(CONFIG["tt_epochs"])):
        model.train()
        total_loss = 0.0
        total_weight = 0.0
        for xb_cat, xb_cont, yb, wb in tr_loader:
            xb_cat = xb_cat.to(device)
            xb_cont = xb_cont.to(device)
            yb = yb.to(device)
            wb = wb.to(device)

            optimizer.zero_grad()
            logits = model(xb_cat, xb_cont)
            per_sample_loss = criterion(logits, yb)         # [B]
            loss = (per_sample_loss * wb).sum() / wb.sum().clamp(min=1e-12)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * float(wb.sum().item())
            total_weight += float(wb.sum().item())

        y_val_true, y_val_score, w_val_b = predict_scores(va_loader)
        val_auc = roc_auc_score(
            y_val_true, y_val_score,
            sample_weight=(w_val_b if split_data.w_val is not None else None),
        )
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
        print(f"Epoch {epoch+1}: train_loss={total_loss/max(total_weight,1e-12):.4f}, val_auc={val_auc:.4f}")
        if patience_count >= patience:
            print("Early stopping.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    y_val_true, y_val_score, w_val_b = predict_scores(va_loader)
    y_test_true, y_test_score, w_test_b = predict_scores(te_loader)

    sw_val = w_val_b if split_data.w_val is not None else None
    sw_test = w_test_b if split_data.w_test is not None else None

    tuned_thr, tuned_metrics_val = find_best_threshold(
        y_val_true, y_val_score, CONFIG["threshold_objective"], sample_weight=sw_val,
    )
    metrics_05 = compute_metrics(y_test_true, y_test_score, 0.5, sample_weight=sw_test)
    metrics_tuned = compute_metrics(y_test_true, y_test_score, tuned_thr, sample_weight=sw_test)
    prob_metrics = compute_prob_metrics(y_test_true, y_test_score, sample_weight=sw_test)

    return {
        "model_name": "tabtransformer",
        "training_paradigm": "supervised",
        "do_hpo": False,
        "best_iteration": None,
        "metrics_0.5": metrics_05,
        "metrics_tuned": metrics_tuned,
        "prob_metrics": prob_metrics,
        "tuned_threshold": tuned_thr,
        "val_best_metrics_at_tuned_threshold": tuned_metrics_val,
        "y_score_test": np.asarray(y_test_score),
    }

In [ ]:
# =========================
# RUN SELECTED MODEL
# =========================
model_name = CONFIG["model_name"].lower()

if model_name == "logreg":
    result = run_logreg_or_xgb("logreg", split_data)
elif model_name == "xgb":
    result = run_logreg_or_xgb("xgb", split_data)
elif model_name == "catboost":
    result = run_catboost(split_data)
elif model_name == "tabtransformer":
    result = run_tabtransformer(split_data)
else:
    raise ValueError(f"Unsupported model_name: {CONFIG['model_name']}")

split_prevalence = {
    "train": float(split_data.y_train.mean()),
    "val": float(split_data.y_val.mean()),
    "test": float(split_data.y_test.mean()),
}

summary = {
    "model_name": result["model_name"],
    "training_paradigm": result["training_paradigm"],
    "do_hpo": result["do_hpo"],
    "best_params": result.get("best_params"),
    "best_cv_score": result.get("best_cv_score"),
    "best_iteration": result.get("best_iteration"),
    "imbalance_mode": CONFIG["imbalance_mode"],
    "preproc_variant": CONFIG["preproc_variant"],
    "split_mode": CONFIG["split_mode"],
    "temporal_test_year": CONFIG.get("temporal_test_year"),
    "use_survey_weights": CONFIG["use_survey_weights"],
    "n_input_features": len(INPUT_FEATURES),
    "input_features": INPUT_FEATURES,
    "split_prevalence": split_prevalence,
    "train_years": split_data.train_years,
    "test_years": split_data.test_years,
    "metrics_0.5": {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in result["metrics_0.5"].items()},
    "metrics_tuned": {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in result["metrics_tuned"].items()},
    "prob_metrics": result["prob_metrics"],
    "tuned_threshold": result["tuned_threshold"],
    "val_best_metrics_at_tuned_threshold": result["val_best_metrics_at_tuned_threshold"],
}
summary

In [ ]:
# =========================
# DISPLAY RESULTS
# =========================
print("=== Training note ngắn ===")
if result["model_name"] == "logreg":
    print("- Paradigm: supervised")
    print("- Train: one-hot + scale, LogisticRegression, class_weight nếu bật native_weight, sample_weight từ _LLCPWT nếu bật")
    print("- Hyper tuning: mặc định KHÔNG")
elif result["model_name"] == "xgb":
    print("- Paradigm: supervised")
    print("- Train: one-hot + XGBoost, scale_pos_weight nếu bật native_weight, sample_weight từ _LLCPWT nếu bật")
    print("- Stopping: early stopping trên val (logloss), patience =", CONFIG["xgb_early_stopping_rounds"])
elif result["model_name"] == "catboost":
    print("- Paradigm: supervised")
    print("- Train: CatBoost với native categorical handling, eval_set=val, sample_weight từ _LLCPWT nếu bật")
    print("- Stopping: use_best_model + early_stopping_rounds=20")
elif result["model_name"] == "tabtransformer":
    print("- Paradigm: supervised")
    print("- Train: categorical embeddings + transformer encoder + MLP head")
    print("- Loss: per-sample BCEWithLogits × sample_weight, pos_weight nếu bật native_weight")
    print("- Seed: torch.manual_seed(RANDOM_STATE) — reproducible")
    print("- Hyper tuning: mặc định KHÔNG, có early stopping theo val AUC")

print("\n=== Split info ===")
print(f"  split_mode        = {CONFIG['split_mode']}")
print(f"  train years       = {split_data.train_years}")
print(f"  test  years       = {split_data.test_years}")
print(f"  use_survey_weights= {CONFIG['use_survey_weights']}")
print(f"  prevalence train  = {split_data.y_train.mean():.4f}")
print(f"  prevalence val    = {split_data.y_val.mean():.4f}")
print(f"  prevalence test   = {split_data.y_test.mean():.4f}")

print("\n=== Metrics @ 0.5 ===")
for k, v in result["metrics_0.5"].items():
    if k != "ConfusionMatrix":
        print(f"  {k:>10}: {float(v):.4f}")

print("\n=== Metrics @ tuned threshold ===")
for k, v in result["metrics_tuned"].items():
    if k != "ConfusionMatrix":
        print(f"  {k:>10}: {float(v):.4f}")

print("\n=== Probability metrics ===")
for k, v in result["prob_metrics"].items():
    print(f"  {k:>12}: {float(v):.4f}")
print(f"  (PR-AUC Lift > 1 nghĩa là model tốt hơn constant predictor; lift = PR_AUC / Prevalence)")

show_confusion_matrix(result["metrics_0.5"]["ConfusionMatrix"], title=f"{result['model_name']} @ 0.5")
show_confusion_matrix(result["metrics_tuned"]["ConfusionMatrix"], title=f"{result['model_name']} @ tuned threshold")
show_calibration(split_data.y_test.to_numpy(), result["y_score_test"], title=f"{result['model_name']} calibration")

In [ ]:
# =========================
# SAVE OUTPUTS
# =========================
outdir = Path(CONFIG["outdir"])
outdir.mkdir(parents=True, exist_ok=True)

with open(outdir / f"summary_{CONFIG['model_name']}_{CONFIG['split_mode']}.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

pd.DataFrame([{
    "model_name": summary["model_name"],
    "training_paradigm": summary["training_paradigm"],
    "do_hpo": summary["do_hpo"],
    "best_iteration": summary.get("best_iteration"),
    "imbalance_mode": summary["imbalance_mode"],
    "preproc_variant": summary["preproc_variant"],
    "split_mode": summary["split_mode"],
    "use_survey_weights": summary["use_survey_weights"],
    "train_years": str(summary["train_years"]),
    "test_years": str(summary["test_years"]),
    "Prevalence_train": summary["split_prevalence"]["train"],
    "Prevalence_val":   summary["split_prevalence"]["val"],
    "Prevalence_test":  summary["split_prevalence"]["test"],
    "Accuracy_0.5":  summary["metrics_0.5"]["Accuracy"],
    "Precision_0.5": summary["metrics_0.5"]["Precision"],
    "Recall_0.5":    summary["metrics_0.5"]["Recall"],
    "F1_0.5":        summary["metrics_0.5"]["F1"],
    "Accuracy_tuned":  summary["metrics_tuned"]["Accuracy"],
    "Precision_tuned": summary["metrics_tuned"]["Precision"],
    "Recall_tuned":    summary["metrics_tuned"]["Recall"],
    "F1_tuned":        summary["metrics_tuned"]["F1"],
    "ROC_AUC":      summary["prob_metrics"]["ROC_AUC"],
    "PR_AUC":       summary["prob_metrics"]["PR_AUC"],
    "PR_AUC_Lift":  summary["prob_metrics"]["PR_AUC_Lift"],
    "Brier":        summary["prob_metrics"]["Brier"],
    "TunedThreshold": summary["tuned_threshold"],
}]).to_csv(outdir / f"metrics_{CONFIG['model_name']}_{CONFIG['split_mode']}.csv", index=False)

print("Saved to:", outdir.resolve())

## Cách dùng notebook này

Chỉ cần đổi các flag trong cell `CONFIG`:
- `model_name`: `"logreg"` / `"xgb"` / `"catboost"` / `"tabtransformer"`
- `split_mode`: `"temporal"` (primary — train 2019–2020, test 2021) hoặc `"random"` (sensitivity check)
- `temporal_test_year`: năm làm test (mặc định 2021)
- `imbalance_mode`: `"none"` hoặc `"native_weight"`
- `preproc_variant`: `"baseline"` / `"missing_indicator"` / `"explicit_unknown"`
- `use_survey_weights`: `True` (suy rộng population, CDC recommendation) / `False` (default, compat với literature ML)

## Protocol baseline chuẩn
- **Primary split**: `temporal` — mô phỏng deployment (train trên quá khứ, test trên năm mới)
- **Sensitivity split**: chạy lại với `random` để so sánh, kiểm tra IID hypothesis
- **Imbalance**: `native_weight` (không sampler); survey weights là riêng (sampling design)
- **Models**: chạy 3 baseline tabular trước (`logreg`, `xgb`, `catboost`), sau mới đến `tabtransformer`
- **Nhánh generative LLM**: để pha sau, không thuộc baseline

## Reporting
- Mỗi run output JSON + CSV vào `./baseline_notebook_outputs/`
- Luôn report cùng **prevalence từng split** và **PR-AUC lift** (= PR_AUC / Prevalence) để chuẩn hóa so sánh qua các split/năm

## Nếu muốn mở rộng sau baseline
- **Group A:** đổi `preproc_variant`
- **Group B:** thêm SMOTE/SMOTENC hoặc focal loss — nhưng không thay baseline mặc định

## Lưu ý khi chạy notebook này

### 1. Trước khi chạy (setup)
- **Packages**: `pip install numpy pandas scikit-learn xgboost catboost torch matplotlib` (nếu thiếu, cell IMPORTS sẽ báo lỗi).
- **Data**: điền đường dẫn CSV vào `CONFIG["y2019"]`, `CONFIG["y2020"]`, `CONFIG["y2021"]`. Có thể chỉ điền 1–2 năm nhưng **temporal split yêu cầu có năm train < `temporal_test_year` và năm test đúng bằng `temporal_test_year`**.
- **Smoke test**: set `CONFIG["max_rows_per_year"] = 20000` để chạy thử ~1 phút trước khi chạy full.
- **Reset state**: nếu bạn đã chạy rồi đổi CONFIG, **Restart Kernel** rồi "Run All" — vì nhiều cell ghi đè globals (`INPUT_FEATURES`, `NOMINAL_COLS`, ...).

### 2. Sanity check sau mỗi bước
| Sau cell | Kiểm tra |
|---|---|
| **Data loading** | `Row attrition per year`: `kept` gần bằng `completed`; `dropped_target_na` nhỏ (<5%). `pos_rate` ~0.05–0.10 cho mọi năm. `weight_var = _LLCPWT`. |
| **Intersection** | `Dropped (not in every year)` nên là `[]` với 2019–2021. Nếu có drop → confirm candidate list còn đúng tên cột. |
| **Split** | Positive rate train/val/test chênh nhau < 0.01 (stratified). Nếu temporal → train_years = [2019, 2020], test_years = [2021]. |
| **Train** | XGB/CatBoost in `best_iteration` hợp lý (không = `n_estimators`-1 nghĩa là chưa early stop → tăng `n_estimators`). TabTransformer train_loss giảm dần, val_auc tăng; early stop trước khi đạt `tt_epochs`. |
| **Evaluate** | Prevalence test ≈ prevalence train (random) hoặc có thể khác chút (temporal). `PR_AUC_Lift > 1` là bắt buộc — nếu ≈ 1 nghĩa là model vô dụng. |

### 3. Đọc kết quả
**Hiểu metrics nào quan trọng**:
- **ROC-AUC**: invariant với prevalence, phản ánh ranking. Mục tiêu BRFSS heart disease thường **0.80–0.85**.
- **PR-AUC Lift**: đáng tin hơn PR-AUC trần vì đã chuẩn hoá theo prevalence. Lift ~4–6× là đạt cho task này.
- **Recall @ tuned threshold**: nếu objective = recall → mục tiêu ~0.70+, chấp nhận precision thấp (sàng lọc y tế).
- **Brier**: càng thấp càng tốt (0 = perfect), phản ánh cả calibration và refinement.
- **Calibration curve**: gần đường diagonal = xác suất dự đoán khớp tần suất thực.

**Metrics KHÔNG nên tin tưởng**:
- **Accuracy @ 0.5**: với prevalence 5–10%, predict all-zero cũng đạt 0.90–0.95. Vô nghĩa.
- **Precision ở ngưỡng quá thấp**: inflate bởi số TP tăng, không phản ánh chất lượng.

### 4. Báo cáo kết quả
Mỗi run auto-save vào `./baseline_notebook_outputs/`:
- `summary_{model}_{split}.json` — full detail (metrics + confusion matrix + config)
- `metrics_{model}_{split}.csv` — 1 row để merge nhiều run

**Ablation protocol tối thiểu** (chạy ≥ 5 run):
1. `xgb` + `temporal` + `native_weight` — baseline chính
2. `xgb` + `random` + `native_weight` — sensitivity check
3. `logreg`, `catboost`, `tabtransformer` — so cross-model
4. `xgb` + `temporal` + `use_survey_weights=True` — ablation weight
5. `xgb` + `temporal` + `imbalance_mode="none"` — ablation imbalance

Sau đó concat các file `metrics_*.csv` lại thành 1 bảng để so sánh:
```python
import pandas as pd, glob
all_runs = pd.concat([pd.read_csv(f) for f in glob.glob("baseline_notebook_outputs/metrics_*.csv")])
all_runs.sort_values("PR_AUC_Lift", ascending=False)
```

### 5. Các lỗi hay gặp
- **`Intersection rỗng`** → FEATURE_CANDIDATES chưa cover biến thể tên cột của năm bạn load. Bổ sung vào `FEATURE_CANDIDATES`.
- **`FileNotFoundError`** → sai path hoặc file chưa giải nén (BRFSS gốc là .XPT, phải convert sang CSV bằng `pyreadstat` hoặc SAS trước).
- **XGB báo `early_stopping_rounds` incompat** → xgboost < 2.0. Upgrade: `pip install -U xgboost`.
- **CatBoost `class_weights` error** → phiên bản cũ dùng `class_weights` như dict; notebook dùng list `[1.0, ratio]` chuẩn của catboost >= 1.0.
- **TabTransformer NaN loss** → thường do `_LLCPWT` có 0 hoặc NaN không được clean. Đã có `_build_weights` fallback 1.0, nhưng kiểm tra lại nếu thấy `val_auc=nan`.
- **RAM lớn** → bật `max_rows_per_year` để smoke test, hoặc giảm `tt_batch_size`.

### 6. Gợi ý khi muốn extend
- **Stratified CV**: thay vì 1 split, `StratifiedKFold(n_splits=5)` → report mean ± std.
- **Bootstrap CI**: resample test 1000 lần, tính 95% CI cho ROC-AUC / PR-AUC Lift.
- **Per-subgroup metrics**: báo cáo riêng cho nam/nữ, nhóm tuổi — quan trọng cho fairness.
- **Feature importance**: `model.feature_importances_` (xgb/catboost), `coef_` (logreg), SHAP cho all.
- **Calibration**: nếu Brier cao / calibration curve lệch → Platt scaling hoặc isotonic regression trên val.

## Training paradigm cụ thể cho từng model

| Hạng mục | **logreg** | **xgb** | **catboost** | **tabtransformer** |
|---|---|---|---|---|
| **Paradigm** | Supervised (linear) | Supervised (gradient boosting) | Supervised (gradient boosting) | Supervised (deep learning) |
| **Kiến trúc** | `LogisticRegression` | `XGBClassifier` | `CatBoostClassifier` | Cat embeddings → `TransformerEncoder` → MLP |
| **Preprocess** | Median impute + `StandardScaler`; OneHot cat | Median impute; OneHot cat (không scale) | Median impute; native cat | Tokenize cat (vocab từ train + `__UNK__`); standardize continuous |
| **Loss** | Binary cross-entropy | Binary cross-entropy (`binary:logistic`) | Binary cross-entropy (`Logloss`) | Binary cross-entropy (`BCEWithLogitsLoss`, weighted-mean per-sample) |
| **Optimizer** | sklearn lbfgs (`max_iter=4000`) | XGBoost (`tree_method="hist"`) | CatBoost ordered boosting | `AdamW(lr=1e-3, wd=1e-5)` |
| **Default params (no HPO)** | `max_iter=4000` | `n_est=1000, lr=0.05, depth=5, subsample=0.8, colsample=0.8` | `iterations=1000, depth=6, lr=0.05` | `d_token=32, n_layers=2, n_heads=4, mlp=64, dropout=0.1, bs=1024, epochs=25` |
| **HPO grid** (`use_hpo=True`) | `C ∈ {0.01, 0.1, 1, 10}` | `depth ∈ {3,5,7}`, `lr ∈ {0.05,0.1}`, `n_est ∈ {300,500}`, `subsample ∈ {0.8,1.0}`, `colsample ∈ {0.8,1.0}` | `depth ∈ {4,6,8}`, `lr ∈ {0.05,0.1}`, `iter ∈ {300,500}` | ❌ (deep learning, quá chậm) |
| **HPO method** | GridSearchCV, 3-fold stratified, scoring=`roc_auc`, refit | ditto | ditto | – |
| **Imbalance** (khi `native_weight`) | `class_weight="balanced"` | `scale_pos_weight=neg/pos` | `class_weights=[1, neg/pos]` | `BCE pos_weight=neg/pos` |
| **Survey weight** (khi `use_survey_weights`) | `sample_weight` trong `.fit()` | `sample_weight` + `sample_weight_eval_set` | `Pool(weight=...)` | per-sample weight × loss |
| **Stopping** (no HPO) | lbfgs convergence | Early stop val (logloss), patience 20 | `use_best_model=True`, patience 20 | Early stop val ROC-AUC, patience 5, max 25 epoch |
| **Stopping** (HPO) | – | Fixed `n_estimators` từ grid | Fixed `iterations` từ grid | – |
| **Threshold** | Quét 0.05–0.95 trên val theo `threshold_objective` | ditto | ditto | ditto |
| **Device** | CPU | CPU (`n_jobs=1` ngoài HPO) | CPU | CUDA / CPU |
| **Seed** | 42 | 42 | 42 | torch + CUDA + numpy + DataLoader generator |

**Common evaluation**:
- Threshold metrics: Acc/Precision/Recall/F1 ở `0.5` + tuned threshold.
- Probability metrics: ROC-AUC, PR-AUC, **Prevalence**, **PR-AUC Lift**, Brier.
- Khi `use_survey_weights=True`, mọi metric tính với `sample_weight=_LLCPWT`.
- Visual: confusion matrix (2 ngưỡng) + calibration curve (10 quantile bins).
- Khi `use_hpo=True`, log thêm `best_params` + `best_cv_score`.